#Rocket Lab Dados - 2026.2#

## <center> Tarefa 1 - Processamento de Dados</center>

## Exercício 1
O Ultimate Team (FUT) é um modo do jogo FIFA no qual o jogador monta seu próprio time adquirindo atletas virtuais.
Cada atleta possui atributos que influenciam seu desempenho em campo — como drible, chute, passe, defesa, velocidade e físico.

Principais colunas:
- `player_id` — identificador único do jogador
- `player_name` — nome do atleta
- `nationality` — país de origem
- `club` — clube atual
- `overall` — nota geral do jogador
- `potential` — potencial máximo de evolução
- `value_eur, wage_eur` — valor de mercado e salário
- `age, height_cm, weight_kg` — características físicas
- `pace, shooting, passing, dribbling, defending, physic` — atributos técnicos

_**Preencha os espacos ____ para carregar os dados e realizar as consultas propostas.**_

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

#caminho do CSV dentro do repositório (notebooks/ e data/ lado a lado)
caminho = "file:" + os.path.abspath("../data/fut_players_data.csv")

### Exercício 1.1 Faça a leitura do arquivo fut_players (fut_player_data.csv) e retorne as 5 primeiras linhas

In [ ]:

# Faça a leitura do arquivo fut_players
fut_players = spark.read.csv(caminho, header=True, inferSchema=True)

# Retorne as 5 primeiras linhas do DF
display(fut_players.limit(5))

### Exercício 1.2 - Retorna a nacionalidade dos jogadores "The Bests"

São considerados jogadores The Bests os que possuem os atributos de drible (_dribbling_) e chute (_shooting_) superior a 90.
Após a geração do DF _The_Best_ realize o join com o df _nationalities_ para obter a nacionalidade dos jogadores.

A sua tabela final deve conter as seguintes informações:
- `player_id`
- `player_name`
- `nationality`
- `position`
- `dribbling`
- `shooting`
- `overall`

In [ ]:
# Aplique os filtros para retornar os jogadores the bests
the_best = (fut_players
            .filter((F.col("dribbling") > 90) & (F.col("shooting") > 90))
            .select("player_id", "player_name", "position", "dribbling", "shooting", "overall"))

# nationalities é um DataDrame da nacionalidade dos jogadores
nationalities = (fut_players.select('player_id', 'player_name', 'nationality'))

# faça um join dos dois DataDrames, mantendo todos os jogadores de the_best e obtendo suas nacionalidades (dica: a chave é o id)
the_best_nationality = (the_best
                        .join(nationalities, on=["player_id", "player_name"], how="left")
                        .select("player_id", "player_name", "nationality", "position",
                                "dribbling", "shooting", "overall"))

display(the_best_nationality)

###  Exercício 2
Use o método *.groupby()* para descobrir qual país tem o melhor *overall* médio. Crie a coluna 'avg_overall'

Seu df country_avg_overall deve conter as seguintes colunas:
- `nationality`
- `overall`
- `avg_overall`

In [ ]:
country_avg_overall = (fut_players
                       .groupBy("nationality")
                       .agg(
                           F.max("overall").alias("overall"),
                           F.mean("overall").alias("avg_overall")
                           )
                       )

# Retornar a nacionalidade com maior overall médio e o overall médio do brasil
melhor = (
    country_avg_overall
    .orderBy(F.col("avg_overall").desc())
    .limit(1)
    .collect()[0]
)

brasil = (
    country_avg_overall
    .filter(F.col("nationality") == "Brazil")
    .collect()[0]
)

display({
    "Melhor overall médio": f"{melhor['nationality']}: {melhor['avg_overall']:.2f}",
    "Overall médio do Brasil": round(brasil['avg_overall'], 2)
})


### Exercício 2.1
Crie um racional que retorne a classificação para o jogador de acordo com as instruções abaixo, então aplique isso para o dataframe fut_players.

*Observação:* considere os limites dentro do intervalo de classificação.
exemplo

-50 contém todos os valores menores que 50 e o valor 50 incluso;


51-60 contém todos os valores entre 51 e 60 com os limites [51,60] inclusos no grupo;


e assim por diante ...

In [ ]:

"""
    Através do overall do jogador retorne a classificação conforme a seguir:
    Overall -> classification
    -50     -> "Amador"
    51-60   -> "Ruim"
    61-70   -> "Ok"
    71-80   -> "Bom"
    81-90   -> "Ótimo"
    91+     -> "Lenda"

    I: int overall
    O: string
"""
# Dica utilize as clasulas when e otherwise
fut_players = (fut_players
               .select('player_id', 'overall')
               .withColumn('classification',
                           F.when(F.col('overall') <= 50, 'Amador')
                            .when(F.col('overall') <= 60, 'Ruim')
                            .when(F.col('overall') <= 70, 'Ok')
                            .when(F.col('overall') <= 80, 'Bom')
                            .when(F.col('overall') <= 90, 'Ótimo')
                            .otherwise('Lenda')))

# Contar quantos jogadores há em cada classificação
fut_players.groupBy("classification").count().orderBy("count", ascending=False).display()

## Desafio — Montando o Time dos Sonhos do 🇧🇷

Ainda utilizando a base **`fut_players_data`**, imagine que você é um grande fã do jogo *FIFA*, e deseja montar o **Time dos Sonhos (Dream Team)** do **Brasil**, selecionando os **melhores jogadores por posição**, ou seja, aqueles com o **maior overall** dentro de cada grupo de posição.

Para isso, adote a **formação tática 4-4-2**, composta por:

- **1 Goleiro (GK)**  
- **4 Defensores (Defesa)**  
- **4 Meio-campistas (Meio)**  
- **2 Atacantes (Ataque)**  

### Objetivo
Criar um *DataFrame* com **11 linhas**, representando o **melhor jogador de cada posição dentro da formação 4-4-2**, com as seguintes colunas:

- `nationality` — nacionalidade do jogador  
- `position_group` — posição agrupada (Goleiro, Defesa, Meio, Ataque)  
- `player_name` — nome do jogador  
- `overall` — nota geral (overall)

---

### Agrupamento de posições
Para facilitar a análise, agrupe as posições originais da base conforme a tabela abaixo:

| **position_group** | **Posições incluídas (`position`)** | **Descrição** |
|:--------------------|:------------------------------------|:---------------|
| **Goleiro** | `GK` | Jogadores que atuam exclusivamente no gol. |
| **Defesa** | `CB`, `LB`, `RB`, `LWB`, `RWB` | Zagueiros e laterais (defensores). |
| **Meio** | `CM`, `CDM`, `CAM`, `LM`, `RM` | Meio-campistas centrais, volantes e meias ofensivos/laterais. |
| **Ataque** | `ST`, `CF`, `LW`, `RW`, `LF`, `RF` | Atacantes e pontas. |
| **Outros** | *(demais posições não classificadas)* | Jogadores fora do esquema tático principal (ex: cartas especiais). |

---

### 🏁 Entrega esperada
Seu *DataFrame final* deve retornar **11 jogadores**, representando o **Time dos Sonhos do Brasil (formação 4-4-2)**, conforme os critérios acima.

In [ ]:
#############################################
########## Construa sua Query Aqui ##########
#############################################

# Realize novamente o import da base
fut_players = spark.read.csv(caminho, header=True, inferSchema=True)

# Só jogadores do Brasil
jogadores_br = fut_players.filter(F.col("nationality") == "Brazil")

# Agrupando as posições
jogadores_br = jogadores_br.withColumn(
    "position_group",
    F.when(F.col("position") == "GK", "Goleiro")
     .when(F.col("position").isin("CB", "LB", "RB", "LWB", "RWB"), "Defesa")
     .when(F.col("position").isin("CM", "CDM", "CAM", "LM", "RM"), "Meio")
     .when(F.col("position").isin("ST", "CF", "LW", "RW", "LF", "RF"), "Ataque")
     .otherwise("Outros")
)

#quantas vagas cada grupo tem na formação 4-4-2
vagas = (F.when(F.col("position_group") == "Goleiro", 1)
          .when(F.col("position_group") == "Defesa", 4)
          .when(F.col("position_group") == "Meio", 4)
          .when(F.col("position_group") == "Ataque", 2)
          .otherwise(0))

# ranking dentro de cada grupo (maior overall primeiro; player_id desempata)
janela = Window.partitionBy("position_group").orderBy(F.col("overall").desc(), F.col("player_id"))

dream_team = (jogadores_br
              .withColumn("ranking", F.row_number().over(janela))
              .filter(F.col("ranking") <= vagas)
              .select("nationality", "position_group", "player_name", "overall")
              .orderBy("position_group", F.col("overall").desc())
              )

display(dream_team)

### Desafio Bônus

Você deve ter notado que **Neymar** aparece tanto entre os melhores jogadores de **ataque** quanto do **meio-campo**.  
Isso acontece porque o dataset contém **múltiplas versões do mesmo jogador**, inclusive atuando em **outras posições**, o que é típico dos modos do *FIFA/Ultimate Team*.

O seu desafio agora é **refazer o exercício anterior**, garantindo que **cada jogador apareça apenas uma vez** no *DataFrame final*.

- Caso o jogador possua mais de uma versão (carta), **considere apenas aquela com o maior valor de `overall`**.  
- Em seguida, **reaplique a lógica da formação 4-4-2**, selecionando os melhores por grupo de posição.

---

### 🏁 Entrega Esperada
Seu *DataFrame final* deve retornar **11 jogadores únicos**, representando o **Dream Team do Brasil** na **formação tática 4-4-2**, **sem repetição de atletas**, conforme os critérios estabelecidos acima.

In [ ]:
#############################################
########## Construa sua Query Aqui ##########
#############################################


# Fica só com a carta de maior overall de cada jogador
janela_jogador = Window.partitionBy("player_name").orderBy(F.col("overall").desc(), F.col("player_id"))

jogadores_unicos = (jogadores_br
                    .withColumn("versao", F.row_number().over(janela_jogador))
                    .filter(F.col("versao") == 1)
                    )

# Mesma lógica (reaproveitar a janela e as vagas)
dream_team_unico = (jogadores_unicos
                    .withColumn("ranking", F.row_number().over(janela))
                    .filter(F.col("ranking") <= vagas)
                    .select("nationality", "position_group", "player_name", "overall")
                    .orderBy("position_group", F.col("overall").desc())
                    )

display(dream_team_unico)